# Saved L2 data to exact strategy P&L

Run the first **10,000 messages of the saved OKX ETH-USDT spot archive for
20 September 2026 (UTC)** through the installed shared implementation:
**saved L2 → EDA → book rule → IOC orders/fills → cash, positions and P&L**.

This fixed engineering sample uses a five-level imbalance rule and a no-trade
control. It does not establish profitability or full-day coverage. Exchange
timestamps are observed; receipt times and sequence continuity remain unknown.
The separately configured feed/processing/order/response delays are assumptions.

**Run All** only reads the selected saved archive and writes a new local attempt.
No source download, live trading, fitting or frozen P1 modification occurs.

This is the fixed [replacement MR8 simulation scope](../docs/research/saved-baseline-acceptance.md).
Historical eligibility, positive coverage and exhaustive correction history remain
unqualified. Frozen P1 stays deferred and unaccepted under its unchanged criteria.

## 1. Choose the saved data and scenario

The versioned backtest configuration owns the instrument, date, 10,000-message
prefix, decision interval, sizing, fees, independent limits and all modeled delays.
The archive path selects one exact completed download. See the
[usage guide](notebook-operations.md#historical-order-book-study) for CLI and recovery.

In [ ]:
# standard library imports
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
CONFIG = root / "configs/backtests/okx-eth-usdt-book-v1.json"
ARCHIVE = root / "data/okx/history/f62e64b7-1b40-4869-8b05-2ca911fefcef"
OUTPUT = root / "logs/book-backtest/notebook"


In [ ]:
# standard library imports
import platform
from fractions import Fraction

# third party library imports
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

# project library imports
import market_data
from accounting import Ledger
from applications.backtest import run_books
from configuration import BookRun
from research.studies import BookReport
from runtime import RuntimeFiles
from serialization import DomainCodec

if platform.python_version() != (root / ".python-version").read_text().strip():
    raise RuntimeError("Select the repository's pinned Python kernel.")
if "site-packages" not in Path(market_data.__file__).resolve().parts:
    raise RuntimeError("Install the repository package before running this notebook.")
installed = dict(RuntimeFiles.inputs())
source = {
    p.relative_to(root / "src").as_posix(): p
    for package in RuntimeFiles.roots
    for p in (root / "src" / package).rglob("*")
    if p.is_file() and (p.suffix in {".py", ".json"} or p.name == "py.typed")
}
if source.keys() != installed.keys() or any(
    p.read_bytes() != installed[name].read_bytes() for name, p in source.items()
):
    raise RuntimeError("The installed package differs from the checkout; reinstall and restart.")
codec = DomainCodec()
config = BookRun.read(CONFIG)
print(config.history.symbol, config.history.day, config.history.max_records, "messages")
print("Modeled timing:", codec.encode_json(config.timing).decode())


## 2. Verify, inspect and simulate through the package

`run_books` is the same orchestrator called by `alpha-backtest books`. It verifies
saved provenance and the complete compressed archive, replays the bounded native
prefix through the existing source reader, computes exact EDA and runs isolated
accounts. It retains decisions, IOC outcomes, per-level fills, valuations,
complete checkpoints, input/config/code identities and failed attempts.

No replay, feature, strategy, fee or accounting formula is copied into this notebook.

In [ ]:
output = run_books(CONFIG, ARCHIVE, output=OUTPUT)
report = BookReport.read(output / "report.json")
print("Completed report:", output / "report.html")
print("Raw SHA-256:", report.raw_sha256)
print("Run identity:", report.run_identity)
print("First / last source times:", report.features[0].source_at, report.features[-1].source_at)
print("Original receipt times known:", sum(row.received_ns is not None for row in report.features))
print("Source sequence IDs known:", sum(row.sequence is not None for row in report.features))


## 3. Inspect the visible book

The shared feature owner measures best quotes, spread and the configured depth
imbalance `(bid size − ask size) / (bid size + ask size)` exactly. These descriptive
rows preserve missing/invalid states. Conversion to floats below serves only the
charts and statistics; the simulation and ledger retain exact units.
Each message has equal weight here, not each elapsed second.

In [ ]:
frame = pd.DataFrame(
    [
        {
            "time_UTC": row.source_at,
            "bid": None if row.bid is None else float(row.bid),
            "ask": None if row.ask is None else float(row.ask),
            "spread": None if row.spread is None else float(row.spread),
            "bid_size": float(row.bid_size),
            "ask_size": float(row.ask_size),
            "imbalance": None if row.imbalance is None else float(
                Fraction(int(row.imbalance.numerator), int(row.imbalance.denominator))
            ),
            "quality": row.reason,
        }
        for row in report.features
    ]
)
display(frame[["spread", "bid_size", "ask_size", "imbalance"]].describe())
display(frame["quality"].value_counts())


## 4. View book conditions and costed NAV

Book charts use observed exchange time. NAV is sampled at modeled decision clocks
and the terminal. Buys walk available asks; sells walk bids. Unfilled IOC quantity
is canceled. A 1% per-level participation cap and persistent consumption prevent
reusing the same displayed liquidity. Source updates at an existing price retain
that consumption; removing the level clears its debt.

The scenario assumes no market impact, 10 bps taker fees rounded per level fill,
100 ms feed delay, 10 ms processing, 20 ms order entry and 100 ms response delay.
Cash and positions stay independent across the rule and control.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True, constrained_layout=True)
axes[0].plot(frame["time_UTC"], frame["bid"], label="Best bid", linewidth=0.8)
axes[0].plot(frame["time_UTC"], frame["ask"], label="Best ask", linewidth=0.8)
axes[0].set_ylabel(f"{config.history.quote_currency} / {config.history.base_currency}")
axes[0].legend(loc="upper left")
axes[0].set_title(f"OKX {config.history.symbol} · {config.history.day} · {len(frame):,} messages")
axes[1].plot(frame["time_UTC"], frame["spread"], color="#94612f", linewidth=0.8)
axes[1].set_ylabel("Spread")
axes[2].plot(frame["time_UTC"], frame["imbalance"], color="#2c7a70", linewidth=0.8)
axes[2].axhline(float(config.instances[0].imbalance_threshold), color="grey", linewidth=0.5)
axes[2].set_ylabel(f"Top-{config.instances[0].depth} imbalance")
axes[2].set_ylim(-1.05, 1.05)
for instance, account in zip(config.instances, report.accounts, strict=True):
    marks = account.state.valuations
    axes[3].plot(
        [row.at for row in marks],
        [None if row.mark is None else float(row.mark.nav.value) for row in marks],
        label=instance.instance,
    )
axes[3].set_ylabel(f"NAV ({config.history.quote_currency})")
axes[3].set_xlabel("UTC")
axes[3].legend(loc="upper left")
for axis in axes:
    axis.grid(alpha=0.2)
fig.savefig(output / "orderbook-and-nav.png", dpi=150)
plt.show()


## 5. Reconcile and interpret the result

The report contains exact cash, holdings, fees, realized/unrealized P&L and NAV.
Reconstruct each terminal balance from all saved fills through the shared `Ledger`;
show both P&L attributions and depth slippage without converting money to floats.
The no-trade control must remain flat with zero fees. Terminal inventory is valued
at a fresh exchange best bid; this is not a simulated liquidation or a depth guarantee.
Unknown source continuity and modeled availability remain limitations even when
accounting reconciles perfectly.

The fixed sample is an engineering demonstration, not a tuned or qualified strategy.
Compare another declared timing scenario only as a separately retained attempt.
The CLI supports complete interruption/resumption with `--stop-at` and `--resume`.
See [the acceptance record](../docs/research/saved-baseline-acceptance.md) for actual results,
fixture proofs, installed-package reproduction and remaining limits.

In [ ]:
summaries = []
for instance, account in zip(config.instances, report.accounts, strict=True):
    terminal = account.state.valuations[-1]
    if terminal.mark is None:
        raise RuntimeError("Completed report has no terminal valuation.")
    restored = Ledger(account.state.ledger)
    if restored.balance != terminal.balance:
        raise RuntimeError("Reopened ledger differs from the recorded terminal balance.")
    summaries.append(
        {
            "instance": instance.instance,
            "decisions": len(account.state.decisions),
            "orders": len(account.state.orders),
            "level_fills": len(account.state.ledger.fills),
            "cash": str(terminal.balance.cash.value),
            "position": str(terminal.balance.holding.value),
            "fees": str(terminal.balance.fees.value),
            "NAV": str(terminal.mark.nav.value),
            "realized_before_fees": str(terminal.balance.realized.value),
            "unrealized_before_fees": str(terminal.mark.unrealized.value),
            "gross_at_reference": str(terminal.mark.gross.value),
            "depth_slippage": str(terminal.balance.slippage.value),
            "net_PnL": str(terminal.mark.net.value),
        }
    )
print("Both exact terminal balances reproduced from every saved fill through the shared ledger.")
display(pd.DataFrame(summaries))
display(HTML((output / "report.html").read_text()))
print("Completed notebook artifacts:", output)
